## 데이터 병합

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 전체 CSV 로딩 (3,753개 파일 머지)
volume_path = "/Volumes/bronze_api/agrofood_risesandfalls/volumn/*.csv"
df = spark.read.option("header", True).option("inferSchema", True).csv(volume_path)

# 가격 관련 컬럼 숫자형으로 변환
price_cols = [c for c in df.columns if 'prc' in c]
numeric_cols = ['ctgry_cd', 'grd_cd', 'item_cd', 'se_cd', 'unit_sz', 'vrty_cd'] + price_cols

df2 = df
for col_name in numeric_cols:
    df2 = df2.withColumn(col_name, F.col(col_name).cast(DoubleType()))

print(f"총 행 수: {df2.count():,}")
print(f"컬럼 수: {len(df2.columns)}")
print()

# 기초통계 (수치형)
display(df2.select(price_cols).describe())

In [0]:
# 컬럼명, null 개수, null 비율, 컬럼 설명 출력
col_desc = {
    'ctgry_cd': '카테고리 코드',
    'ctgry_nm': '카테고리명',
    'item_cd': '품목 코드',
    'item_nm': '품목명',
    'grd_cd': '등급 코드',
    'grd_nm': '등급명',
    'se_cd': '구분 코드',
    'se_nm': '구분명',
    'unit_sz': '단위 크기',
    'unit_nm': '단위명',
    'vrty_cd': '품종 코드',
    'vrty_nm': '품종명',
    'exmn_ymd': '조사일',
    'exmn_dd_avg_prc': '일 평균가격',
    'exmn_dd_cnvs_avg_prc': '일 평균가격(비교용)',
    'dd1_bfr_cmpr_rafrt': '전일 대비 등락률',
    'ww1_bfr_cmpr_rafrt': '전주 대비 등락률',
    'mm1_bfr_cmpr_rafrt': '전월 대비 등락률',
    'yy1_bfr_cmpr_rafrt': '전년 대비 등락률'
}

total = df2.count()
null_counts = df2.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df2.columns]
).toPandas().T
null_counts.columns = ['null_count']
null_counts['null_pct'] = (null_counts['null_count'] / total * 100).round(2)
null_counts['desc'] = [col_desc.get(c, '') for c in null_counts.index]
display(null_counts.reset_index().rename(columns={'index': 'col'}))

In [0]:
# 컬럼명, null 개수, null 비율, 결측치 아닌 개수, 컬럼 설명 출력
col_desc = {
    'ctgry_cd': '카테고리 코드',
    'ctgry_nm': '카테고리명',
    'item_cd': '품목 코드',
    'item_nm': '품목명',
    'grd_cd': '등급 코드',
    'grd_nm': '등급명',
    'se_cd': '구분 코드',
    'se_nm': '구분명',
    'unit_sz': '단위 크기',
    'unit_nm': '단위명',
    'vrty_cd': '품종 코드',
    'vrty_nm': '품종명',
    'exmn_ymd': '조사일',
    'exmn_dd_avg_prc': '일 평균가격',
    'exmn_dd_cnvs_avg_prc': '일 평균가격(비교용)',
    'dd1_bfr_cmpr_rafrt': '전일 대비 등락률',
    'ww1_bfr_cmpr_rafrt': '전주 대비 등락률',
    'mm1_bfr_cmpr_rafrt': '전월 대비 등락률',
    'yy1_bfr_cmpr_rafrt': '전년 대비 등락률'
}

total = df2.count()
null_counts = df2.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df2.columns]
).toPandas().T
null_counts.columns = ['null_count']
null_counts['notnull_count'] = total - null_counts['null_count']
null_counts['null_pct'] = (null_counts['null_count'] / total * 100).round(2)
null_counts['desc'] = [col_desc.get(c, '') for c in null_counts.index]
display(null_counts.reset_index().rename(columns={'index': 'col'}))

In [0]:
for c in df2.columns:
    null_rows = df2.filter(F.col(c).isNull())
    if null_rows.count() > 0:
        print(f"\n컬럼: {c} (null값 {null_rows.count()}개)")
        display(null_rows)

In [0]:
cat_cols = ['ctgry_nm', 'item_nm', 'vrty_nm', 'grd_nm', 'se_nm', 'unit']
print("범주형 컬럼 고유값 수")
for c in cat_cols:
    cnt = df2.select(c).distinct().count()
    print(f"  {c}: {cnt}개")

display(df2.groupBy('ctgry_cd','ctgry_nm').count().orderBy(F.desc('count'))) # 부류명
display(df2.groupBy('item_cd','item_nm').count().orderBy(F.desc('count')))  # 품목명
display(df2.groupBy('vrty_cd','vrty_nm').count().orderBy(F.desc('count')))  # 품종명
display(df2.groupBy('grd_cd','grd_nm').count().orderBy(F.desc('count')))   # 등급명
display(df2.groupBy('se_cd','se_nm').count().orderBy(F.desc('count')))    # 구분명
display(df2.groupBy('unit').count().orderBy(F.desc('count')))     # 단위